In [1]:
!pip install -q kagglehub scikit-image opencv-python-headless onnxruntime pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 48.8 MB/s eta 0:00:00


In [2]:
# upload the onnx models
from google.colab import files
import os

MODEL_FILES = {
    "LBP + KNN": "lbp_knn.onnx",
    "HOG + Linear SVM": "hog_svm.onnx",
    "mini-Xception": "mini_xception.onnx",
}

print({f: os.path.exists(f) for f in MODEL_FILES.values()})

{'lbp_knn.onnx': True, 'hog_svm.onnx': True, 'mini_xception.onnx': True}


In [3]:
import kagglehub, numpy as np, cv2

# download the dataset FER-2013
path = kagglehub.dataset_download("msambare/fer2013")
test_dir = os.path.join(path, "test")

emotion_map = {
    'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3,
    'neutral': 4, 'sad': 5, 'surprise': 6
}
id_to_emotion = {v: k for k, v in emotion_map.items()}

# select 50 test images
N_IMAGES = 50
rng = np.random.RandomState(42)

paths, labels = [], []
for name, idx in emotion_map.items():
    folder = os.path.join(test_dir, name)
    for f in os.listdir(folder):
        paths.append(os.path.join(folder, f)); labels.append(idx)
paths, labels = np.array(paths), np.array(labels)

sel = rng.choice(len(paths), size=N_IMAGES, replace=False)
test_imgs, test_labels = [], []
for i in sel:
    im = cv2.imread(paths[i], cv2.IMREAD_GRAYSCALE)
    if im.shape != (48, 48):
        im = cv2.resize(im, (48, 48), interpolation=cv2.INTER_AREA)
    test_imgs.append(im); test_labels.append(labels[i])
test_labels = np.array(test_labels)

print(len(test_imgs), "images, class counts:", np.bincount(test_labels, minlength=7))

Using Colab cache for faster access to the 'fer2013' dataset.
50 images, class counts: [11  1  5 13  8  8  4]


In [4]:
from skimage.feature import local_binary_pattern

# feature extraction lbp (must match function used in training)
def lbp_features(image, num_points=16, radius=2):

    lbp = local_binary_pattern(image, num_points, radius, method='uniform')
    n_bins = num_points + 2 # histogram bins

    grid_x, grid_y = 6, 6 # divides images into a grid
    h, w = image.shape
    cell_h, cell_w = h // grid_y, w // grid_x # calculates block size

    feats = []
    for i in range(grid_y):
        for j in range(grid_x):
            block = lbp[i * cell_h:(i + 1) * cell_h, j * cell_w:(j + 1) * cell_w]
            hist, _ = np.histogram(block.ravel(), bins=n_bins, range=(0, n_bins))
            feats.append(hist)
    feats = np.concatenate(feats)
    return feats

In [5]:
from skimage.feature import hog

# feature extraction hog (params match those on the training phase)
hog_params = dict(
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
)

def hog_features(image):
    return hog(image, **hog_params).astype(np.float32)

In [7]:
import onnxruntime as ort

# makes inference sessions for tests
def make_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 1
    return ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])

feature_fns = {
    "LBP + KNN": lambda im: lbp_features(im).astype(np.float32)[None, :],
    "HOG + Linear SVM": lambda im: hog_features(im)[None, :],
    "mini-Xception": lambda im: (im.astype(np.float32) / 255.0)[None, :, :, None],
}

sessions = {name: make_session(f) for name, f in MODEL_FILES.items()}

In [25]:
import time, pandas as pd

def get_scores(outputs):
    for o in outputs:
        o = np.asarray(o)
        if o.ndim == 2 and np.issubdtype(o.dtype, np.floating):
            return o

N_WARMUP = 5
rows = []

for name, sess in sessions.items():
    fn = feature_fns[name]
    in_name = sess.get_inputs()[0].name

    # warm up session, does not enter the results
    for _ in range(N_WARMUP):
        sess.run(None, {in_name: fn(test_imgs[0])})

    for img, y in zip(test_imgs, test_labels):
        t0 = time.perf_counter()
        x = fn(img)
        t1 = time.perf_counter()
        out = sess.run(None, {in_name: x})
        t2 = time.perf_counter()

        scores = get_scores(out)[0]
        rows.append({
            "model": name,
            "true": y,
            "pred": int(scores.argmax()),
            "features_ms": (t1 - t0) * 1000,
            "inference_ms": (t2 - t1) * 1000,
            "total_ms": (t2 - t0) * 1000,
        })

df = pd.DataFrame(rows)
df["correct"] = df["true"] == df["pred"]

In [26]:
summary = df.groupby("model", sort=False).agg(
    accuracy=("correct", "mean"),
    features_ms=("features_ms", "mean"),
    inference_ms=("inference_ms", "mean"),
    total_ms=("total_ms", "mean")
).round(3)

print(f"{N_IMAGES} test images")
print(summary)

50 test images
                  accuracy  features_ms  inference_ms  total_ms
model                                                          
LBP + KNN             0.36        4.203       120.947   125.150
HOG + Linear SVM      0.50        0.934         0.045     0.979
mini-Xception         0.68        0.012         0.323     0.335
